# RiskOptima Dislocation Model

This notebook demonstrates `riskoptima.dislocation`. Equities use quality, valuation and a simplified FCF-proxy scenario; funds use historical price/regime dislocation without corporate DCF or a claimed NAV discount. Scores are research heuristics, not calibrated investment forecasts.


### Execution environment

Run from the repository root with the checkout installed (`python -m pip install -e .`). By default this notebook fetches Yahoo Finance data. Set `RISKOPTIMA_DEMO_LIVE=0` before execution for explicitly synthetic, deterministic inputs. This is never a silent fallback for failed live data. Individual ticker failures appear as ERROR rows; an entirely unsuccessful scan fails the notebook.


In [ ]:
from IPython.display import Markdown, display
import pandas as pd
import os
from examples.example_dislocation_model import SyntheticProvider

from riskoptima.dislocation import (
    DislocationConfig,
    analyze_symbol,
    generate_markdown_report,
    scan_universe,
)

pd.set_option("display.max_columns", 30)
USE_LIVE_DATA = os.environ.get("RISKOPTIMA_DEMO_LIVE", "1") != "0"
provider = None if USE_LIVE_DATA else SyntheticProvider()
print("Data source:", "Yahoo Finance (live)" if USE_LIVE_DATA else "Synthetic, not market data")


## Input universe

The symbols below are Yahoo identifiers, not ISINs. Yahoo's ISIN search resolves IE00B50W2R13 to `0P0000KM24.L` (Global Bond Index GBP-hedged accumulation share class). Verify share class and currency before replacing an identifier.


In [ ]:
INSTRUMENTS = [
    ("0P0000KM24.L", "Global Bond Index Fund GBP Hedged Acc"),
    ("0P0000TKZI.L", "LifeStrategy 40% Equity Fund"),
    ("0P0000TKZO.L", "LifeStrategy 100% Equity Fund"),
    ("0P0000TKZM.L", "LifeStrategy 80% Equity Fund"),
    ("0P00017UWT.L", "Target Retirement 2045 Fund"),
    ("0P0000UGLG.L", "ESG Developed World All Cap Equity Index Fund"),
    ("0P0000UGLE.L", "ESG Developed Europe Index Fund"),
    ("0P0000KM22.L", "Emerging Markets Stock Index Fund"),
    ("0P0000KM20.L", "Pacific ex-Japan Stock Index Fund"),
    ("0P00018XAP.L", "FTSE 100 Index Unit Trust"),
    ("0P0000KSPA.L", "U.S. Equity Index Fund"),
]

pd.DataFrame(INSTRUMENTS, columns=["symbol", "name"])


## Run and rank all instruments

The engine combines price drawdown, trend deviation, volatility and regime diagnostics. Data coverage measures input completeness, not model accuracy. Markov fitting uses a fixed seed; a fit failure is recorded in diagnostics before the momentum/volatility fallback is used.


In [ ]:
config = DislocationConfig(history_period="5y", min_observations=126, use_markov_regime=True)
summary, reports = scan_universe(INSTRUMENTS, config=config, provider=provider, continue_on_error=True)
assert reports, "No successful analyses; inspect the ticker error rows"
print(f"Successful analyses: {len(reports)}/{len(INSTRUMENTS)}")
display(summary.loc[summary["classification"] == "ERROR", ["symbol", "error"]])
summary


## Generate the full report for every successful ticker

The default prose is generated from the measured state and is reproducible. `generate_markdown_report` also accepts a `narrative_generator` callback, so an LLM can be injected by an application without adding an LLM SDK dependency to RiskOptima.


In [ ]:
generated_reports = {report.symbol: generate_markdown_report(report) for report in reports}

for symbol, report_markdown in generated_reports.items():
    display(Markdown(report_markdown))
    display(Markdown("---"))


## Single-ticker equity example

For MSFT, equity metadata activates fundamental scoring. The simplified DCF discounts the supplied FCF proxy, without normalized FCFE or net borrowing. Margin of safety is `(model_value - price) / model_value`; upside divides by price instead. Annualized convergence assumes the current model value is reached over the selected horizon, excludes distributions, and is not a return forecast. Missing cash flow or shares results in unavailable DCF fields, not a fabricated value.


In [ ]:
equity_report = analyze_symbol("MSFT", config=config, provider=provider)
assert equity_report.instrument_type == "EQUITY"
if equity_report.intrinsic_value is not None:
    assert abs(equity_report.margin_of_safety - (1 - equity_report.price / equity_report.intrinsic_value)) < 1e-10
display(Markdown(generate_markdown_report(equity_report)))
payload = equity_report.as_dict()
assert "riskoptima_version" in payload


## Optional LLM narrative adapter

Applications can inject any callable accepting the report context and returning text. This keeps the quantitative model deterministic while allowing the final narrative to be genuinely generative.

```python
def my_generator(context: dict) -> str:
    # Call your preferred model/service here.
    return "..."

report = analyze_symbol("MSFT", narrative_generator=my_generator)
```
